In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, date_format, current_timestamp

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:
df = spark.table("bronze.boe_mortgage_approvals")

df_silver = (
    df
    .withColumn("report_date", to_date(col("DATE"), "dd MMM yyyy"))
    .withColumn("year_month", date_format(col("report_date").cast("timestamp"), "yyyy-MM"))
    .withColumnRenamed("LPMVTVN", "mortgage_approvals_count")
    .withColumn("mortgage_approvals_count", col("mortgage_approvals_count").cast("long"))
    .drop("DATE")
    .select("report_date", "year_month", "mortgage_approvals_count")
)

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.boe_mortgage_approvals")
)

In [0]:
df_silver.show(5)
df_silver.printSchema()
spark.sql("SELECT MIN(report_date), MAX(report_date), COUNT(*) FROM silver.boe_mortgage_approvals")

In [0]:
# silver.boe_mortgage_approvals
# 267 rows, Jan 2004 to Mar 2026
# UK-wide mortgage approvals count, monthly
# report_date (date), year_month (string), mortgage_approvals_count (long)